# 03 · 베이스라인 학습과 성능 지표

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-axis/Wind-Power-Forecasting/blob/main/notebooks/03_baseline_training_and_metrics.ipynb)

현재 구현된 14개 베이스라인과 EMFN, 총 15개 모델 계열을 동일 평가 시점에 재실험합니다. 6개 시계(1/3/6/9/12/24h), 학습 시드 42/2026/3407을 사용합니다. Persistence와 ARIMA는 시드 반복을 만들지 않습니다. 모델별 입력 예산과 출력 형태는 아래 표로 구분합니다. 과거 v1/v2와 구조 어블레이션은 이 베이스라인 목록에 포함하지 않습니다.

In [1]:
from pathlib import Path
import os, sys

# Google Colab 자동 감지 및 환경 설정
if 'google.colab' in sys.modules or (os.path.exists('/content') and not (Path.cwd() / "configs/base_config.yaml").exists()):
    repo_dir = Path('/content/Wind-Power-Forecasting')
    if not repo_dir.exists():
        print("Google Colab 환경 감지: Wind-Power-Forecasting 저장소를 클론합니다...")
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lab-axis/Wind-Power-Forecasting.git", str(repo_dir)], check=True)
        print("필수 의존 패키지를 설치합니다 (colab/requirements-colab.txt)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "colab/requirements-colab.txt")], check=True)
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    try:
        get_ipython().run_line_magic('cd', str(repo_dir))
    except Exception:
        pass

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/base_config.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
runtime = ROOT / ".experiment_archive/python_runtime"
if runtime.exists(): sys.path.insert(0, str(runtime))
from IPython.display import display
from src.workflows import research as wf
%matplotlib inline

In [2]:
wf.model_catalog()

,model,input,output,source
0,EMFN (Proposed),24h generation + six weather channels,Bernoulli–Beta,src/models/emfn.py
1,DLinear (+Weather),24h generation + six weather channels,point (Dirac for CRPS),src/models/dlinear.py
2,LSTM (+Weather),24h generation + six weather channels,point (Dirac for CRPS),src/models/rnn.py
3,GRU (+Weather),24h generation + six weather channels,point (Dirac for CRPS),src/models/rnn.py
4,CNN-LSTM (+Weather),24h generation + six weather channels,point (Dirac for CRPS),src/models/cnn_lstm.py
5,TCN (+Weather),24h generation + six weather channels,point (Dirac for CRPS),src/models/tcn_model.py
6,Transformer (+Weather),24h generation + six weather channels,point (Dirac for CRPS),src/models/transformer.py
7,LightGBM (24h matched),same flattened 24h × 7 inputs,point (Dirac for CRPS),src/models/probabilistic_lightgbm.py
8,Quantile LightGBM (24h matched),same flattened 24h × 7 inputs,quantile distribution,src/models/probabilistic_lightgbm.py
9,Hurdle Quantile LightGBM (24h matched),same flattened 24h × 7 inputs,quantile distribution,src/models/probabilistic_lightgbm.py


## 1. 고정된 학습 규약

설정: `configs/all_baselines.yaml`, 실행: `src/experiments/run_all_baselines.py`, 생성자: `src/models/benchmark_registry.py`. 신경망 체크포인트와 트리 반복 수, ARIMA 차수는 검증 CRPS로 선택합니다. 결정론 CRPS는 clipped MAE입니다. 학습 손실은 EMFN NLL+Huber, 다른 신경망 Huber, 점예측 LightGBM L2를 유지합니다.

GRU의 마지막 ReLU는 제거해 음수 초기 출력에서도 학습 기울기가 흐르도록 수정했습니다. 새 결과는 수정 전 0 수렴 실행과 구분합니다. 장기 lag LightGBM은 NaN을 자체 처리하고 최대 168h lag·달력 특징을 사용하므로 24h 입력 모델과 구조만의 비교가 아닙니다. Prophet은 기존 연간/주간/일간 계절성 설정의 train-only 달력 예측이며 rolling refit을 하지 않습니다. 학습 자료가 2년 미만이므로 연간 계절성 식별의 한계도 있습니다.

In [3]:
wf.run_status()

,run_id,status,models,local_artifacts_available,model_horizon_seed_combinations,metric_rows,seeds,horizons,test_role,data_sha256
0,all_baselines_20261005T143556+0900,complete,15,True,234,468,"[42, 2026, 3407]","[1, 3, 6, 9, 12, 24]",development_benchmark_previously_consulted,01861a5a18cac3731f4238d9307d03e2bb29d9c3aa11e8...


## 2. 재학습 실행 (선택)

기본값 False는 검증된 저장 결과를 읽는 모드입니다. True이면 **EMFN을 포함한 전체 행렬을 처음부터** 새 run 폴더에 학습하며 상당한 시간이 걸립니다. 234는 모델×시계×시드 조합 수이지 독립 적합 함수 호출 수가 아닙니다. ARIMA 후보 5개와 시드별 Prophet은 한 번 적합한 파라미터를 여러 시계에서 사용합니다. 완료 후 검증기를 실행해야 최신 manifest와 보고 표로 승격됩니다.

In [4]:
RUN_FULL_RETRAINING = False
if RUN_FULL_RETRAINING:
    new_run = wf.train_all()
    wf.publish_run(new_run)

## 3. 검증에 의한 선택과 학습 실패 진단

아래 표의 검증 예측 표준편차가 0이면 상수 예측 경고입니다. 경고를 숨기거나 2025 성능을 보고 좋은 시드만 고르지 않습니다. 선택 없는 Persistence/Prophet은 그 사실을 그대로 기록합니다.

In [5]:
wf.validation_diagnostics()

,model,horizon_hours,seed,selection_basis,best_epoch,stopped_epoch,best_val_crps,validation_raw_std_mwh,validation_clipped_std_mwh,validation_constant_warning
0,Naive Persistence,1,NaN,fixed_rule_no_selection,NaN,NaN,NaN,4.265190,4.265190,False
1,Diurnal Persistence,1,NaN,fixed_rule_no_selection,NaN,NaN,NaN,4.223567,4.223567,False
2,Naive Persistence,3,NaN,fixed_rule_no_selection,NaN,NaN,NaN,4.265699,4.265699,False
3,Diurnal Persistence,3,NaN,fixed_rule_no_selection,NaN,NaN,NaN,4.229473,4.229473,False
4,Naive Persistence,6,NaN,fixed_rule_no_selection,NaN,NaN,NaN,4.263940,4.263940,False
...,...,...,...,...,...,...,...,...,...,...
229,Transformer (+Weather),24,3407.0,validation_crps,7.0,12.0,2.610647,2.160014,2.159840,False
230,LightGBM (24h matched),24,3407.0,validation_crps,NaN,NaN,2.868876,1.667933,1.667933,False
231,Quantile LightGBM (24h matched),24,3407.0,validation_crps,NaN,NaN,1.816902,1.343934,1.341778,False
232,Hurdle Quantile LightGBM (24h matched),24,3407.0,validation_crps,NaN,NaN,1.818218,1.359811,1.360999,False


## 4. 베이스라인 결과

CRPS ↓, 영발전 Brier ↓, RMSE ↓, MAE ↓, 영발전 AUPRC ↑ 순서입니다. 확률 모델의 RMSE는 예측 평균, MAE는 예측 중앙값입니다. 점예측 모델의 CRPS는 MAE이고 확률 지표는 N/A입니다. 표시한 ±는 시드별 지표의 표본 표준편차이며 앙상블 점수가 아닙니다. 단일 결정론 실행의 SD는 계산하지 않습니다.

In [6]:
baselines = [name for name in wf.ALL_MODELS if name != "EMFN (Proposed)"]
wf.display_scores(models=baselines, horizons=(1, 6, 24))

,model,horizon_hours,n_fits,crps,zero_brier,rmse,mae,zero_auprc
0,DLinear (+Weather),1,3,0.8461 ± 0.0225,N/A,1.3288 ± 0.0256,0.8461 ± 0.0225,N/A
1,LSTM (+Weather),1,3,0.7899 ± 0.0147,N/A,1.2806 ± 0.0244,0.7900 ± 0.0147,N/A
2,GRU (+Weather),1,3,0.8038 ± 0.0111,N/A,1.3023 ± 0.0163,0.8038 ± 0.0111,N/A
3,CNN-LSTM (+Weather),1,3,0.8591 ± 0.0222,N/A,1.3810 ± 0.0286,0.8591 ± 0.0222,N/A
4,TCN (+Weather),1,3,0.8557 ± 0.0611,N/A,1.3594 ± 0.0923,0.8557 ± 0.0611,N/A
5,Transformer (+Weather),1,3,0.8603 ± 0.0134,N/A,1.3681 ± 0.0286,0.8603 ± 0.0135,N/A
6,LightGBM (24h matched),1,3,0.8209 ± 0.0000,N/A,1.2867 ± 0.0000,0.8209 ± 0.0000,N/A
7,Quantile LightGBM (24h matched),1,3,0.5960 ± 0.0000,0.1179 ± 0.0000,1.3950 ± 0.0000,0.8051 ± 0.0000,0.5986 ± 0.0000
8,Hurdle Quantile LightGBM (24h matched),1,3,0.5813 ± 0.0000,0.0643 ± 0.0000,1.3489 ± 0.0000,0.7921 ± 0.0000,0.7585 ± 0.0000
9,LightGBM (+Weather),1,3,0.7371 ± 0.0000,N/A,1.1716 ± 0.0000,0.7371 ± 0.0000,N/A


모든 시계·시드·구현 지표를 읽으려면 아래 DataFrame을 사용합니다. 파일: `reports/tables/all_baselines_metrics.csv`(468행), `all_baselines_summary.csv`(180행). Notebook의 접힌 표시가 결과 누락을 뜻하지 않습니다.

In [7]:
wf.metric_definitions()

,metric,unit_direction,interpretation
0,crps,MWh; lower,"Full bounded distribution; 1001-point grid, Di..."
1,zero_brier / zero_auprc / zero_auroc / zero_ece,lower / higher / higher / lower,Observed Y=0 event; N/A without fitted probabi...
2,rmse / mse,MWh / MWh²; lower,Predictive mean for distributions; clipped poi...
3,mae,MWh; lower,Predictive median for distributions; clipped p...
4,mae_mean,MWh; lower,"Per-run CSV: MAE of predictive mean, NOT seed ..."
5,nmae_pct / nrmse_pct,% of C; lower,Both use predictive MEAN for distributions; nm...
6,rse / rrse / rae,dimensionless; lower,Relative point errors against reference variab...
7,wape_pct / smape_pct / mape_nonzero_pct,%; lower,Mean point forecast; nonzero MAPE excludes exa...
8,r2 / corr / mbe / tic,dimensionless except MBE (MWh),"Point fit, correlation, signed bias and Theil ..."
9,mae_ramp / rmse_ramp / corr_ramp / vpr_pct,MWh / MWh / r / %,Consecutive 1h pairs only; VPR is prediction/o...


In [8]:
all_baseline_metrics = wf.metrics_table(models=baselines, horizons=None, full=True)
display(all_baseline_metrics)
print("Available metric columns:", all_baseline_metrics.columns.tolist())

,model,seed,horizon_hours,split,selection_basis,mae,rmse,mse,nmae_pct,nrmse_pct,...,crps,zero_point_f1_mean,zero_point_balanced_acc_mean,zero_brier,zero_auprc,zero_auroc,zero_ece,picp_90_pct,pinaw_90_pct,winkler_score_90
1,Naive Persistence,NaN,1,test,fixed_rule_no_selection,0.8602,1.4337,2.0555,4.0963,6.8272,...,0.860221,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Diurnal Persistence,NaN,1,test,fixed_rule_no_selection,3.5634,5.4338,29.5260,16.9686,25.8752,...,3.563410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Naive Persistence,NaN,3,test,fixed_rule_no_selection,1.7905,2.8251,7.9814,8.5263,13.4530,...,1.790521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Diurnal Persistence,NaN,3,test,fixed_rule_no_selection,3.5636,5.4342,29.5310,16.9695,25.8773,...,3.563597,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Naive Persistence,NaN,6,test,fixed_rule_no_selection,2.3944,3.8407,14.7507,11.4017,18.2889,...,2.394366,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459,Transformer (+Weather),3407.0,24,test,validation_crps,3.1064,4.8002,23.0418,14.7922,22.8580,...,3.106366,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
461,LightGBM (24h matched),3407.0,24,test,validation_crps,3.3772,4.3946,19.3125,16.0818,20.9267,...,3.377186,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
463,Quantile LightGBM (24h matched),3407.0,24,test,validation_crps,3.0006,4.3667,19.0682,16.0353,20.7939,...,2.162791,0.0,0.5,0.139754,0.209070,0.595393,0.074659,90.66,54.30,16.8024
465,Hurdle Quantile LightGBM (24h matched),3407.0,24,test,validation_crps,3.0170,4.3817,19.1997,16.1124,20.8654,...,2.175792,0.0,0.5,0.132950,0.242116,0.651681,0.021330,90.76,55.08,16.4622


Available metric columns: ['model', 'seed', 'horizon_hours', 'split', 'selection_basis', 'mae', 'rmse', 'mse', 'nmae_pct', 'nrmse_pct', 'rse', 'rrse', 'rae', 'wape_pct', 'smape_pct', 'mape_nonzero_pct', 'r2', 'corr', 'mbe', 'tic', 'count', 'mae_ramp', 'rmse_ramp', 'corr_ramp', 'vpr_pct', 'bound_violation_pct', 'integrated_negative_mwh', 'min_pred_mwh', 'max_pred_mwh', 'zero_f1', 'zero_balanced_acc', 'imbalance_loss_mwh', 'mae_cut_in', 'rmse_cut_in', 'cut_in_count', 'mae_mean', 'crps', 'zero_point_f1_mean', 'zero_point_balanced_acc_mean', 'zero_brier', 'zero_auprc', 'zero_auroc', 'zero_ece', 'picp_90_pct', 'pinaw_90_pct', 'winkler_score_90']


**다음:** [04 · EMFN 구조와 학습](04_emfn_architecture_and_results.ipynb).